# 01 · Ingestion
PDF text extraction with OCR fallback (whole scanned slides + screenshots/tables inside slides), header/footer removal, slide-title and code detection.

Put PDFs in `data/pdfs/<role>/` where role is `textbook`, `pyq` or `syllabus`. Output: `data/ingested/<source_id>.json`.

**Before running:** the vision model is `qwen3-vl:4b-instruct-q4_K_M`; make sure no other model is loaded (`ollama ps`; stop one with `ollama stop <name>`) — the vision model should have the GPU to itself during ingestion.

## 1 · Imports and settings

In [1]:
import base64, hashlib, json, re, statistics, time, difflib
from collections import Counter
from pathlib import Path

import pymupdf as fitz                       # PyMuPDF
import requests
from rapidocr_onnxruntime import RapidOCR

DATA_DIR = Path("../data")
PDF_DIR = DATA_DIR / "pdfs"                  # subfolders = roles
OUT_DIR = DATA_DIR / "ingested"
for d in (PDF_DIR / "textbook", PDF_DIR / "pyq", PDF_DIR / "syllabus", OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

MIN_CHARS_FOR_TEXT_PAGE = 40                 # fewer characters than this -> treat page as scanned
OCR_DPI = 200
OCR_IMAGE_MIN_AREA = 0.15                    # OCR images covering >=15% of a slide (screenshots, tables)
MIN_OCR_CHARS = 20                           # ignore OCR results shorter than this (logos, icons)

# Vision OCR: RapidOCR is the fast first pass; pages it reads badly go to a local vision model.
OLLAMA_URL = "http://localhost:11434"
VISION_MODEL = "qwen3-vl:4b-instruct-q4_K_M"  # ~3 GB; leaves VRAM headroom on 8 GB
USE_VISION_OCR = True                        # False -> RapidOCR only
VISION_CONF_THRESHOLD = 0.90                 # RapidOCR mean confidence below this -> vision model


## 2 · Helpers: hashing, line joining, code patterns

In [2]:
BULLETS = ("•", "▪", "◦", "‣", "●", "○", "■", "□", "➢", "➤", "►", "✓", "✔", "–", "- ", "* ")
CODE_LINE_RE = re.compile(
    r"^\s*(>>>|\.\.\.\s|def |class |import |from \S+ import|return\b|print\(|"
    r"for .+:$|if .+:$|while .+:$|elif .+:$|else:$|try:$|except\b.*:$|with .+:$)")


def file_hash(path):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def weighted_median(values, weights):
    pairs = sorted(zip(values, weights))
    total, acc = sum(weights), 0
    for v, w in pairs:
        acc += w
        if acc >= total / 2:
            return v
    return pairs[-1][0] if pairs else 0


def starts_new_line(t):
    """Bullets and interactive-prompt code lines keep their own line instead of being joined."""
    t = t.lstrip()
    return t.startswith(BULLETS) or t.startswith((">>>", "...")) or bool(re.match(r"^\(?\d+[.)]\s", t))


def join_lines(lines):
    """Join wrapped lines; keep bullets/code prompts on new lines; undo line-end hyphenation."""
    out = []
    for t in lines:
        t = t.strip()
        if not t:
            continue
        if not out or starts_new_line(t) or out[-1].startswith((">>>", "...")):
            out.append(t)
        elif out[-1].endswith("-") and t[:1].islower():
            out[-1] = out[-1][:-1] + t
        else:
            out[-1] += " " + t
    return "\n".join(out)


def looks_like_code(lines):
    lines = [l for l in lines if l.strip()]
    if not lines:
        return False
    hits = sum(bool(CODE_LINE_RE.match(l)) for l in lines)
    return hits >= max(1, (len(lines) + 1) // 2) and not lines[0].lstrip().startswith(BULLETS)


## 3 · Text-layer extraction

In [3]:
MONO_HINTS = ("mono", "courier", "consol", "menlo", "inconsolata", "code")


def _is_mono(span):
    # PyMuPDF span flag bit 3 (value 8) = monospaced font
    return bool(span["flags"] & 8) or any(h in span["font"].lower() for h in MONO_HINTS)


def rebuild_code(lines, size, char_ratio=0.6):
    """PDFs often drop leading spaces in code; recover indentation from each line's x position."""
    char_w = size * char_ratio
    min_x = min(x for x, _ in lines)
    out = []
    for x, t in lines:
        lead = len(t) - len(t.lstrip(" "))
        out.append(" " * (round((x - min_x) / char_w) + lead) + t.strip())
    return "\n".join(out)


def extract_page_blocks(page):
    """Text blocks (text, size, code/bold flags, position) + rectangles of images on the page."""
    blocks, images = [], []
    for b in page.get_text("dict")["blocks"]:
        if b.get("type") == 1:               # image block
            images.append(tuple(b["bbox"]))
            continue
        if b.get("type") != 0:
            continue
        lines, sizes, weights = [], [], []
        mono = bold = total = 0
        for line in b["lines"]:
            text = "".join(s["text"] for s in line["spans"])
            if not text.strip():
                continue
            for s in line["spans"]:
                n = len(s["text"].strip())
                total += n
                sizes.append(s["size"])
                weights.append(max(n, 1))
                if _is_mono(s):
                    mono += n
                if s["flags"] & 16:          # bit 4 = bold
                    bold += n
            lines.append((line["bbox"][0], text))
        if not lines or total == 0:
            continue
        size = weighted_median(sizes, weights)
        raw = [t for _, t in lines]
        mono_font = mono / total > 0.6
        is_code = mono_font or looks_like_code(raw)
        if is_code:
            text = rebuild_code(lines, size, 0.6 if mono_font else 0.5)
        else:
            text = join_lines(raw)
        blocks.append({"text": text, "size": round(size, 1), "is_code": is_code, "is_table": False,
                       "bold": bold / total > 0.6,
                       "y0": round(b["bbox"][1], 1), "y1": round(b["bbox"][3], 1)})
    return blocks, images


## 4 · OCR fallback (RapidOCR)
Whole page when the slide has no text layer; only the image region for screenshots/tables on normal slides. Boxes on the same row are joined with ` | ` so tables keep their rows.

In [4]:
_OCR = None


def page_needs_ocr(blocks):
    return sum(len(b["text"].strip()) for b in blocks) < MIN_CHARS_FOR_TEXT_PAGE


def ocr_result_to_blocks(result, dpi, offset=(0, 0), col_gap=1.5):
    """RapidOCR [[box, text, score], ...] -> blocks in PDF points.

    Boxes on the same row are joined with ' | ' only when the gap between them is wide
    (> col_gap x line height), i.e. separate table columns; otherwise with a space.
    offset = top-left of the clipped region when OCR ran on part of a page.
    """
    scale, (ox, oy) = 72 / dpi, offset
    lines = []
    for box, text, score in result or []:
        xs = [p[0] for p in box]
        ys = [p[1] for p in box]
        lines.append({"x0": min(xs) * scale + ox, "y0": min(ys) * scale + oy,
                      "y1": max(ys) * scale + oy, "text": text.strip(), "score": float(score)})

    # 1) rows: boxes whose vertical centres are within half a line height of each other
    lines.sort(key=lambda l: (l["y0"] + l["y1"]) / 2)
    rows = []
    for l in lines:
        c, h = (l["y0"] + l["y1"]) / 2, l["y1"] - l["y0"]
        if rows and abs(c - rows[-1]["c"]) < 0.5 * h:
            rows[-1]["items"].append(l)
        else:
            rows.append({"c": c, "items": [l]})
    row_lines = []
    for r in rows:
        items = sorted(r["items"], key=lambda l: l["x0"])
        text, cols = items[0]["text"], 1
        for prev, cur in zip(items, items[1:]):
            h = cur["y1"] - cur["y0"]
            prev_x1 = prev["x0"] + len(prev["text"]) * h * 0.5      # approx right edge
            if cur["x0"] - prev_x1 > col_gap * h:
                text, cols = text + " | " + cur["text"], cols + 1
            else:
                text += " " + cur["text"]
        row_lines.append({"x0": items[0]["x0"], "y0": min(i["y0"] for i in items),
                          "y1": max(i["y1"] for i in items), "text": text,
                          "score": sum(i["score"] for i in items) / len(items), "cols": cols})

    # 2) blocks: consecutive rows with similar height and a small gap
    groups = []
    for l in row_lines:
        h = l["y1"] - l["y0"]
        prev = groups[-1][-1] if groups else None
        if prev and (l["y0"] - prev["y1"]) < 1.3 * h and abs(h - (prev["y1"] - prev["y0"])) < 0.4 * h:
            groups[-1].append(l)
        else:
            groups.append([l])

    blocks = []
    for g in groups:
        texts = [l["text"] for l in g]
        heights = [l["y1"] - l["y0"] for l in g]
        table = sum(l["cols"] > 1 for l in g) >= max(2, len(g) // 2)
        is_code = not table and looks_like_code(texts)
        if is_code:
            char_w = statistics.median(heights) * 0.5
            min_x = min(l["x0"] for l in g)
            text = "\n".join(" " * round((l["x0"] - min_x) / char_w) + l["text"] for l in g)
        elif table:
            text = "\n".join(texts)
        else:
            text = join_lines(texts)
        blocks.append({"text": text, "size": round(statistics.median(heights) / 1.2, 1),
                       "is_code": is_code, "is_table": table, "bold": False,
                       "y0": round(g[0]["y0"], 1), "y1": round(g[-1]["y1"], 1),
                       "ocr_conf": round(sum(l["score"] for l in g) / len(g), 3)})
    return blocks


VISION_PROMPT = (
    "Transcribe ALL text in this image exactly as written. Rules:\n"
    "- If the image is a slide, put its title on the first line starting with '# '.\n"
    "- Put any code inside ```python fences and keep its exact indentation.\n"
    "- Write tables as Markdown tables.\n"
    "- Read multi-column layouts column by column, left column first.\n"
    "- Output only the transcription, no comments."
)


def vision_ocr(png_bytes, model=VISION_MODEL):
    """Local vision model on Ollama -> markdown transcription."""
    r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=600, json={
        "model": model, "stream": False, "keep_alive": "5m",
        "options": {"temperature": 0, "num_ctx": 4096},
        "messages": [{"role": "user", "content": VISION_PROMPT,
                      "images": [base64.b64encode(png_bytes).decode()]}]})
    r.raise_for_status()
    return r.json()["message"]["content"]


def markdown_to_blocks(md, y0=0.0, y1=0.0):
    """Vision output -> blocks. '# ' line = title, ``` fences = code, '|' lines = table."""
    blocks, para, code, table, in_code = [], [], [], [], False

    def flush_para():
        if para:
            blocks.append({"text": join_lines(para), "is_code": False, "is_table": False})
            para.clear()

    def flush_table():
        rows = [r for r in table if not re.fullmatch(r"\|?[\s:|-]+\|?", r.strip())]  # drop |---|
        if rows:
            blocks.append({"text": "\n".join(" | ".join(c.strip() for c in r.strip().strip("|").split("|"))
                                             for r in rows), "is_code": False, "is_table": True})
        table.clear()

    for line in md.splitlines():
        if line.strip().startswith("```"):
            if in_code:
                if code:
                    blocks.append({"text": "\n".join(code).rstrip(), "is_code": True, "is_table": False})
                code.clear()
            else:
                flush_para(); flush_table()
            in_code = not in_code
        elif in_code:
            code.append(line)
        elif line.strip().startswith("|"):
            flush_para(); table.append(line)
        elif line.startswith("# ") and not blocks and not para:
            flush_table()
            blocks.append({"text": line[2:].strip(), "is_code": False, "is_table": False, "is_title": True})
        elif not line.strip():
            flush_para(); flush_table()
        else:
            flush_table(); para.append(line.lstrip("#").strip() if line.startswith("#") else line)
    if in_code and code:
        blocks.append({"text": "\n".join(code).rstrip(), "is_code": True, "is_table": False})
    flush_para(); flush_table()

    step = (y1 - y0) / max(len(blocks), 1)
    for i, b in enumerate(blocks):                 # spread blocks over the region, keep order
        b.update({"size": 0.0, "bold": False, "y0": round(y0 + i * step, 1),
                  "y1": round(y0 + (i + 1) * step, 1), "ocr": "vision"})
    return blocks


def needs_vision(blocks):
    if not blocks:
        return True
    conf = sum(b["ocr_conf"] for b in blocks) / len(blocks)
    return conf < VISION_CONF_THRESHOLD or any(b["is_code"] or b["is_table"] for b in blocks)


def ocr_page(page, dpi=OCR_DPI, clip=None):
    """OCR the whole page, or only the rectangle `clip` (e.g. a screenshot on a slide).
    RapidOCR first; low-confidence / code / table results are redone by the vision model."""
    global _OCR
    if _OCR is None:
        _OCR = RapidOCR()                    # loads once, ~2 s
    rect = fitz.Rect(clip) if clip else page.rect
    png = page.get_pixmap(dpi=dpi, clip=rect).tobytes("png")
    result, _ = _OCR(png, use_cls=False)     # slides are never rotated; the angle classifier flips text
    blocks = ocr_result_to_blocks(result, dpi, offset=(rect.x0, rect.y0))
    if USE_VISION_OCR and needs_vision(blocks):
        try:
            vblocks = markdown_to_blocks(vision_ocr(png), rect.y0, rect.y1)
            if vblocks:
                if clip:                     # image region: the slide title comes from the text layer
                    for b in vblocks:
                        b.pop("is_title", None)
                return vblocks
        except requests.RequestException as e:
            print(f"  vision OCR failed on page {page.number + 1}: {e} -> keeping RapidOCR text")
    return blocks


## 5 · Cleaning: headers/footers and titles

In [5]:
PAGE_NUM_RE = re.compile(r"(page\s*)?#(\s*(of|/)\s*#)?")


def _norm(text):
    return re.sub(r"\d+", "#", text.lower()).strip()


def strip_headers_footers(pages, min_share=0.3, margin=0.08, max_len=120):
    """Drop short blocks in the top/bottom 8% of the page that repeat across pages, and page numbers."""
    def in_margin(b, h):
        return b["y1"] <= h * margin or b["y0"] >= h * (1 - margin)

    n = len(pages)
    counts = Counter()
    for p in pages:
        counts.update({_norm(b["text"]) for b in p["blocks"]
                       if in_margin(b, p["height"]) and len(b["text"]) <= max_len})
    repeated = {t for t, c in counts.items() if n >= 6 and c / n >= min_share}

    removed = []
    for p in pages:
        keep = []
        for b in p["blocks"]:
            t = _norm(b["text"])
            if in_margin(b, p["height"]) and (t in repeated or PAGE_NUM_RE.fullmatch(t)):
                removed.append(b["text"])
            else:
                keep.append(b)
        p["blocks"] = keep
    return pages, removed


def merge_wrapped_titles(blocks, idxs):
    """Merge consecutive title blocks (a title wrapped over two lines) into the first one."""
    if len(idxs) < 2:
        return blocks
    first = blocks[idxs[0]]
    first["text"] = " ".join(blocks[i]["text"].strip() for i in idxs)
    first["y1"] = blocks[idxs[-1]]["y1"]
    drop = set(idxs[1:])
    return [b for i, b in enumerate(blocks) if i not in drop]


def tag_headings(pages, ratio=1.15, max_len=150):
    """Slides: the title is the largest text near the top of the slide.
    Documents: short, not code, and clearly larger than body text (or a bold one-liner)."""
    body = [(b["size"], len(b["text"])) for p in pages for b in p["blocks"]
            if not b["is_code"] and b["size"] > 0]
    if not body:
        return pages, 0
    body_size = weighted_median([s for s, _ in body], [w for _, w in body])

    for p in pages:
        bl = p["blocks"]
        for b in bl:
            b["is_heading"] = False
        if not bl:
            continue
        if any(b.get("is_title") for b in bl):     # title already given by the vision model
            for b in bl:
                b["is_heading"] = bool(b.get("is_title"))
        elif p.get("is_slide"):
            top = [i for i, b in enumerate(bl) if b["y0"] < p["height"] * 0.3
                   and not b["is_code"] and not b.get("is_table") and b["size"] > 0]
            if not top:
                continue
            biggest = max(bl[i]["size"] for i in top)
            title_idx = [i for i in top if bl[i]["size"] >= biggest * 0.9 and len(bl[i]["text"]) <= max_len]
            # keep only the first contiguous run
            run = [title_idx[0]] if title_idx else []
            for i in title_idx[1:]:
                if i == run[-1] + 1:
                    run.append(i)
            if run and biggest >= body_size * 1.05:
                p["blocks"] = bl = merge_wrapped_titles(bl, run)
                bl[run[0]]["is_heading"] = True
        else:
            for b in bl:
                t = b["text"].strip()
                short = 0 < len(t) <= max_len and "\n" not in t
                bigger = b["size"] >= body_size * ratio
                bold_line = b.get("bold", False) and len(t) <= 80
                b["is_heading"] = (not b["is_code"]) and short and (bigger or bold_line)
    return pages, body_size


## 6 · Ingest a PDF / pasted text, save and load

In [6]:
def ingest_pdf(path, role="textbook", use_ocr=True):
    """role: textbook | note | web | pyq | syllabus"""
    path = Path(path)
    t0 = time.time()
    pages, ocr_pages, image_ocr_pages = [], [], []
    with fitz.open(path) as doc:
        for i, page in enumerate(doc):
            blocks, images = extract_page_blocks(page)
            if use_ocr and page_needs_ocr(blocks):
                blocks = ocr_page(page)                        # scanned / image-only slide
                if blocks:
                    ocr_pages.append(i + 1)
            elif use_ocr and images:
                area = page.rect.width * page.rect.height
                for r in images:                               # screenshots of code, tables, diagrams
                    if (r[2] - r[0]) * (r[3] - r[1]) >= OCR_IMAGE_MIN_AREA * area:
                        extra = ocr_page(page, clip=r)
                        if sum(len(b["text"]) for b in extra) >= MIN_OCR_CHARS:
                            for b in extra:
                                b["from_image"] = True
                            blocks.extend(extra)
                            image_ocr_pages.append(i + 1)
                blocks.sort(key=lambda b: b["y0"])
            pages.append({"page_no": i + 1,                    # PDF page index (what a viewer opens)
                          "label": page.get_label(),           # printed page number, if any
                          "height": page.rect.height,
                          "is_slide": page.rect.width > page.rect.height,
                          "blocks": blocks})
    pages, removed = strip_headers_footers(pages)
    pages, body_size = tag_headings(pages)
    h = file_hash(path)
    return {"source_id": h[:12], "name": path.name, "role": role, "hash": h,
            "n_pages": len(pages), "ocr_pages": ocr_pages,
            "vision_pages": sorted({p["page_no"] for p in pages for b in p["blocks"] if b.get("ocr") == "vision"}),
            "image_ocr_pages": sorted(set(image_ocr_pages)), "body_font_size": body_size,
            "removed_headers": sorted(set(removed))[:30],
            "seconds": round(time.time() - t0, 1), "pages": pages}


def ingest_text(title, text, role="note"):
    """Pasted text -> same shape as a PDF with one page; paragraphs become blocks."""
    blocks = []
    for para in (p.strip() for p in re.split(r"\n\s*\n", text)):
        if not para:
            continue
        heading = para.startswith("#") and "\n" not in para
        lines = para.splitlines()
        is_code = looks_like_code(lines)
        blocks.append({"text": para.lstrip("# ").strip() if heading else para, "size": 10.0,
                       "is_code": is_code, "is_table": False, "bold": False,
                       "is_heading": heading, "y0": 0, "y1": 0})
    h = hashlib.sha1(text.encode("utf-8")).hexdigest()
    return {"source_id": h[:12], "name": title, "role": role, "hash": h, "n_pages": 1,
            "ocr_pages": [], "image_ocr_pages": [], "body_font_size": 10.0,
            "removed_headers": [], "seconds": 0,
            "pages": [{"page_no": 1, "label": "", "height": 0, "is_slide": False, "blocks": blocks}]}


def save_doc(doc, out_dir=OUT_DIR):
    path = Path(out_dir) / f"{doc['source_id']}.json"
    path.write_text(json.dumps(doc, ensure_ascii=False, indent=1), encoding="utf-8")
    return path


def load_doc(source_id, out_dir=OUT_DIR):
    return json.loads((Path(out_dir) / f"{source_id}.json").read_text(encoding="utf-8"))


---
# Checks
## 7 · Ingest one deck

In [7]:
pdfs = sorted(PDF_DIR.glob("*/*.pdf"))
print([f"{p.parent.name}/{p.name}" for p in pdfs])

CHECK_PDF = PDF_DIR / "textbook" / "Ch3. Control Flow_part1.pdf"   # change to any deck
doc = ingest_pdf(CHECK_PDF, role=CHECK_PDF.parent.name)
{k: v for k, v in doc.items() if k != "pages"}


['textbook/Ch 1 Data Mining Basics.pdf', 'textbook/Ch 1 Machine Learning.pdf', 'textbook/Ch 3. Classification.pdf', 'textbook/Ch 4 Unsupervised Learning.pdf', 'textbook/Ch 5 File Handling.pdf', 'textbook/Ch 7. Regular Expressions.pdf', 'textbook/Ch1 and 2 Introduction to Python and Python Data Structure.pdf', 'textbook/Ch3 Functions Modules and Packages_part2.pdf', 'textbook/Ch3. Control Flow_part1.pdf', 'textbook/Perceptron and LMS.pdf']


{'source_id': '7d23475b1157',
 'name': 'Ch3. Control Flow_part1.pdf',
 'role': 'textbook',
 'hash': '7d23475b1157fcd9b51a4a9ee706108f7b487652',
 'n_pages': 47,
 'ocr_pages': [4, 5, 6, 16, 18, 19, 20, 21, 23, 25, 27, 34, 35, 37, 39, 42],
 'vision_pages': [4,
  5,
  6,
  13,
  18,
  19,
  20,
  21,
  23,
  25,
  26,
  27,
  28,
  34,
  35,
  37,
  40,
  42],
 'image_ocr_pages': [13, 26, 28, 40],
 'body_font_size': 32.0,
 'removed_headers': [],
 'seconds': 141.4}

## 8 · Look at pages
`H` title, `C` code, `T` table; `vis` = read by the vision model, `img` = RapidOCR on an image region.

Expected on Control Flow: **p.4** table with `a, b = 'good', 'bad'`; **p.10** one title; **p.11** `>>>` lines separate; **p.19 / p.23** IDE screenshots transcribed as real code with indentation.

In [8]:
def show_page(doc, page_no, max_chars=300):
    p = doc["pages"][page_no - 1]
    print(f"--- page {page_no} (label {p['label']!r}), {len(p['blocks'])} blocks ---")
    for b in p["blocks"]:
        tag = "H" if b.get("is_heading") else "C" if b["is_code"] else "T" if b.get("is_table") else " "
        src = "vis" if b.get("ocr") == "vision" else "img" if b.get("from_image") else ""
        print(f"[{tag}] {b['size']:>5} {src:3} | {b['text'][:max_chars]}")
    print()

for n in (4, 10, 11, 19, 23):
    show_page(doc, n)


--- page 4 (label ''), 2 blocks ---
[H]   0.0 vis | Python statements
[T]   0.0 vis | Statement | Role | Example
Assignment | Creating references | a, b = 'good', 'bad'
Calls and other expressions | Running functions | log.write("spam, ham")
print calls | Printing objects | print('The Killer', joke)
if/elif/else | Selecting actions | if "python" in text: <br>    print(text)
for/else 

--- page 10 (label ''), 8 blocks ---
[H]  44.0     | Extended sequence
[C]  32.1     | >>> seq = [1, 2, 3, 4]
>>> a, *b = seq
[ ]  32.0     | a➔1 ,  b➔[2 , 3, 4]
[C]  32.0     | >>>*a, b = seq
[ ]  32.0     | a➔[1, 2 , 3]  ,  b➔4
[C]  32.0     | >>> a, *b, c = seq
a ➔1, b ➔[2, 3] c ➔4
[ ]  28.0     | • A single starred name, *X, can be used in the assignment target in order to specify a more general matching against the sequence e.g. List
[ ]  28.0     | • Collects all items in the sequence not assigned to other names

--- page 11 (label ''), 3 blocks ---
[H]  36.0     | Multiple target assignment and sha

## 9 · Titles and code across the deck

In [9]:
headings = [(p["page_no"], b["text"]) for p in doc["pages"] for b in p["blocks"] if b.get("is_heading")]
codes = [(p["page_no"], b["text"]) for p in doc["pages"] for b in p["blocks"] if b["is_code"]]
tables = [p["page_no"] for p in doc["pages"] for b in p["blocks"] if b.get("is_table")]
print(f"{doc['n_pages']} pages | {len(headings)} titles | {len(codes)} code blocks | tables on {sorted(set(tables))}")
print(f"whole-page OCR: {doc['ocr_pages']}\nimage-region OCR: {doc['image_ocr_pages']}\nvision model used on: {doc['vision_pages']}\n")
for pg, t in headings[:25]:
    print(pg, "|", t)
print("\n--- first 3 code blocks ---")
for pg, t in codes[:3]:
    print(f"[p.{pg}]\n{t}\n")


47 pages | 42 titles | 29 code blocks | tables on [4, 5, 6, 13]
whole-page OCR: [4, 5, 6, 16, 18, 19, 20, 21, 23, 25, 27, 34, 35, 37, 39, 42]
image-region OCR: [13, 26, 28, 40]
vision model used on: [4, 5, 6, 13, 18, 19, 20, 21, 23, 25, 26, 27, 28, 34, 35, 37, 40, 42]

2 | Objective
3 | Python Program Conceptual Hierarchy
4 | Python statements
5 | Python statements
6 | Python statements
7 | Assignment Statements
8 | Assignment Statements
9 | Assignment Statements
10 | Extended sequence
11 | Multiple target assignment and shared references
12 | Variable Naming Convention
13 | Expression Statements
14 | Print operation
15 | Print operation
17 | Python Flow Control
18 | Syntax of if...elif...else
19 | if...elif...else
20 | Nested if statements
22 | For loop
24 | The range() function
25 | Range Example
26 | Range Example
27 | break and continue
28 | for loop withelse
29 | Nested Loops

--- first 3 code blocks ---
[p.6]
| raise | Triggering exceptions | `raise EndSearch(location)` |
| asser

## 10 · What the header/footer remover took out

In [10]:
doc["removed_headers"]


[]

## 11 · OCR test
Turns one text slide into an image (a fake scan) and checks OCR recovers it.

In [11]:
def make_mixed_test_pdf(src, out, scan_page, dpi=150):
    """Copy pages 1..scan_page+2 of a text PDF, replacing scan_page with an image of itself."""
    with fitz.open(src) as s, fitz.open() as new:
        for i in range(min(scan_page + 2, len(s))):
            if i + 1 == scan_page:
                pg = s[i]
                pix = pg.get_pixmap(dpi=dpi)
                img_page = new.new_page(width=pg.rect.width, height=pg.rect.height)
                img_page.insert_image(img_page.rect, pixmap=pix)
            else:
                new.insert_pdf(s, from_page=i, to_page=i)
        new.save(out)
    return Path(out)


def page_text(d, n):
    return "\n".join(b["text"] for b in d["pages"][n - 1]["blocks"])


SCAN_PAGE = 10                                   # a normal text slide
mixed = ingest_pdf(make_mixed_test_pdf(CHECK_PDF, DATA_DIR / "mixed_test.pdf", SCAN_PAGE))
print("OCR pages:", mixed["ocr_pages"], f"(should include {SCAN_PAGE})")
ratio = difflib.SequenceMatcher(None, page_text(doc, SCAN_PAGE), page_text(mixed, SCAN_PAGE)).ratio()
print(f"OCR vs text-layer similarity: {ratio:.2f}")
show_page(mixed, SCAN_PAGE)


OCR pages: [4, 5, 6, 10] (should include 10)
OCR vs text-layer similarity: 0.93
--- page 10 (label ''), 3 blocks ---
[H]   0.0 vis | Extended sequence
[C]   0.0 vis | >>> seq = [1, 2, 3, 4]
>>> a, *b = seq
a→ 1, b→[2 , 3, 4]

>>> *a, b = seq
a→ [1, 2 , 3] , b→4

>>> a, *b, c = seq
a → 1, b → [2, 3] c → 4
[ ]   0.0 vis | - A single starred name, *X, can be used in the assignment target in order to specify a more general matching against the sequence e.g. List
- Collects all items in the sequence not assigned to other names



## 12 · Ingest everything and save

In [12]:
for p in pdfs:
    d = ingest_pdf(p, role=p.parent.name)
    save_doc(d)
    n_titles = sum(b.get("is_heading", False) for pg in d["pages"] for b in pg["blocks"])
    n_code = sum(b["is_code"] for pg in d["pages"] for b in pg["blocks"])
    print(f"{p.parent.name}/{p.name}: {d['n_pages']} pages, {n_titles} titles, {n_code} code, "
          f"OCR {len(d['ocr_pages'])} pages + {len(d['image_ocr_pages'])} image regions "
          f"(vision on {len(d['vision_pages'])}), {d['seconds']} s")


textbook/Ch 1 Data Mining Basics.pdf: 84 pages, 82 titles, 0 code, OCR 4 pages + 8 image regions (vision on 11), 62.0 s
textbook/Ch 1 Machine Learning.pdf: 31 pages, 30 titles, 0 code, OCR 4 pages + 0 image regions (vision on 4), 30.8 s
textbook/Ch 3. Classification.pdf: 194 pages, 178 titles, 1 code, OCR 28 pages + 21 image regions (vision on 45), 322.9 s
textbook/Ch 4 Unsupervised Learning.pdf: 104 pages, 94 titles, 11 code, OCR 17 pages + 10 image regions (vision on 25), 236.7 s
textbook/Ch 5 File Handling.pdf: 40 pages, 38 titles, 29 code, OCR 3 pages + 2 image regions (vision on 4), 25.5 s
textbook/Ch 7. Regular Expressions.pdf: 19 pages, 18 titles, 14 code, OCR 2 pages + 0 image regions (vision on 1), 9.0 s
textbook/Ch1 and 2 Introduction to Python and Python Data Structure.pdf: 104 pages, 97 titles, 43 code, OCR 22 pages + 13 image regions (vision on 33), 231.7 s
textbook/Ch3 Functions Modules and Packages_part2.pdf: 55 pages, 54 titles, 134 code, OCR 4 pages + 3 image regions (

## 13 · Pasted text

In [13]:
note = ingest_text("Decorators", """# Decorators

A decorator is a function that takes another function and extends it.

def deco(f):
    return f""")
print([(b["text"][:30], b["is_heading"], b["is_code"]) for b in note["pages"][0]["blocks"]])
save_doc(note)


[('Decorators', True, False), ('A decorator is a function that', False, False), ('def deco(f):\n    return f', False, True)]


WindowsPath('../data/ingested/15ad4d65e0c2.json')